# Notebook 100 — Setup e ingesta del dataset AgentEval

En esta serie construyes un sistema **RAG** (Retrieval-Augmented Generation) completo sobre el
lakehouse: desde los datos crudos hasta un archivo de submission listo para la competencia de
Kaggle *[AgentEval Part I: Grounded RAG Benchmark](https://www.kaggle.com/competitions/agent-eval-part-i-grounded-rag-benchmark/overview)*. 

El benchmark no evalúa solo si la respuesta es correcta. Evalúa tres cosas que importan en
producción:

| Requisito | Qué significa |
|---|---|
| **Grounding** | La respuesta debe estar respaldada por evidencia real del corpus |
| **Citación correcta** | El sistema debe señalar *cuáles* chunks usó |
| **Control de alucinaciones** | Afirmar cosas sin respaldo tiene costo |

## Qué hace este notebook

1. Verifica que los CSV de Kaggle estén en el Volume de Unity Catalog
2. Los lee con la configuración correcta para texto con comas y saltos de línea
3. Normaliza `gold_chunk_ids` de string a `ARRAY<STRING>`
4. Escribe tres tablas Delta y habilita Change Data Feed en el corpus
5. Hace un EDA breve y crea el split `dev` / `holdout`


## 1. Configuración

In [0]:
CATALOG = "big_data_ii_2025"
SCHEMA = "spark_examples"
VOLUME = "agenteval"
VOL = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}"

# Tablas de la serie
T_CORPUS = f"{CATALOG}.{SCHEMA}.agenteval_corpus"
T_TRAIN = f"{CATALOG}.{SCHEMA}.agenteval_train"
T_TEST = f"{CATALOG}.{SCHEMA}.agenteval_test"
T_DEV = f"{CATALOG}.{SCHEMA}.agenteval_train_dev"
T_HOLDOUT = f"{CATALOG}.{SCHEMA}.agenteval_train_holdout"

SEED = 42  # semilla fija: el split debe ser reproducible entre corridas

print(f"Volume : {VOL}")
print(f"Corpus : {T_CORPUS}")

## 2. Crear catálogo, schema y volume

In [0]:
spark.sql(f"CREATE CATALOG IF NOT EXISTS {CATALOG}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG}.{SCHEMA}.{VOLUME}")

spark.sql(f"USE CATALOG {CATALOG}")
spark.sql(f"USE SCHEMA {SCHEMA}")

print("Catálogo, schema y volume listos.")

## 3. Verificar que los archivos estén subidos

Si esta celda falla, el problema no está en el código: falta subir los archivos por la UI.
El mensaje de error te dice exactamente qué hacer.

In [0]:
REQUIRED_FILES = [
    "corpus.csv",
    "train.csv",
    "test.csv",
    "sample_submission.csv",
]
OPTIONAL_FILES = ["metaData.csv"]

try:
    present = {f.name for f in dbutils.fs.ls(VOL)}
except Exception as e:
    raise RuntimeError(
        f"No se pudo listar el volume {VOL}.\n"
        f"Verifica en Catalog Explorer que el volume '{VOLUME}' exista dentro de "
        f"{CATALOG}.{SCHEMA}.\nDetalle: {e}"
    )

missing = [f for f in REQUIRED_FILES if f not in present]
if missing:
    raise FileNotFoundError(
        f"Faltan archivos en {VOL}: {missing}\n\n"
        "Cómo resolverlo:\n"
        "  1. Descarga los CSV desde la página de la competencia en Kaggle.\n"
        "  2. En el menú lateral: Catalog -> big_data_ii_2025 -> spark_examples -> Volumes -> agenteval\n"
        "  3. Botón 'Upload to this volume' y selecciona los archivos faltantes.\n"
        f"\nArchivos actualmente presentes: {sorted(present)}"
    )

print(f"Archivos requeridos presentes: {REQUIRED_FILES}")
for f in OPTIONAL_FILES:
    print(f"  {f}: {'presente' if f in present else 'ausente (opcional)'}")

## 4. Inspección cruda antes de parsear


In [0]:
def peek(filename: str, n_chars: int = 700) -> None:
    """Imprime los primeros caracteres crudos de un archivo del volume."""
    with open(f"{VOL}/{filename}", "r", encoding="utf-8") as fh:
        raw = fh.read(n_chars)
    print(f"===== {filename} =====")
    print(raw)
    print("...\n")

peek("corpus.csv")
peek("train.csv")

## 5. Lectura de los CSV

Tres opciones son obligatorias para este dataset:

- `header=True` — la primera fila trae los nombres de columna
- `multiLine=True` — `chunk_text` y `answer` contienen **saltos de línea dentro del campo**;
  sin esta opción Spark corta las filas donde no debe y obtienes basura silenciosa
- `escape='"'` — comillas dobles escapadas dentro de campos entrecomillados

Usamos schema explícito en lugar de `inferSchema`: es más rápido (evita una pasada extra sobre
los datos) y sobre todo es determinístico.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType

CSV_OPTS = dict(header=True, multiLine=True, escape='"', quote='"', mode="PERMISSIVE")

schema_corpus = StructType([
    StructField("chunk_id", StringType()),
    StructField("doc_id", StringType()),
    StructField("title", StringType()),
    StructField("doc_type", StringType()),
    StructField("chunk_text", StringType()),
])

schema_train = StructType([
    StructField("question_id", StringType()),
    StructField("question", StringType()),
    StructField("answer", StringType()),
    StructField("gold_chunk_ids", StringType()),  # se parsea a array más adelante
])

schema_test = StructType([
    StructField("question_id", StringType()),
    StructField("question", StringType()),
])

raw_corpus = spark.read.csv(f"{VOL}/corpus.csv", schema=schema_corpus, **CSV_OPTS)
raw_train = spark.read.csv(f"{VOL}/train.csv", schema=schema_train, **CSV_OPTS)
raw_test = spark.read.csv(f"{VOL}/test.csv", schema=schema_test, **CSV_OPTS)

print(f"corpus : {raw_corpus.count():>6,} chunks")
print(f"train  : {raw_train.count():>6,} preguntas")
print(f"test   : {raw_test.count():>6,} preguntas")

### Validación de integridad de la lectura

Si `multiLine` estuviera mal configurado, verías `chunk_id` nulos o con texto pegado. Estos
asserts fallan de inmediato en vez de dejar que el error se propague hasta el índice vectorial.

In [0]:
n_bad_chunk_id = raw_corpus.filter(
    F.col("chunk_id").isNull() | (F.length("chunk_id") > 80)
).count()
assert n_bad_chunk_id == 0, (
    f"{n_bad_chunk_id} filas del corpus tienen chunk_id nulo o sospechosamente largo. "
    "Suele indicar que el CSV se partió mal: revisa multiLine/escape."
)

n_dup = raw_corpus.count() - raw_corpus.select("chunk_id").distinct().count()
assert n_dup == 0, f"Hay {n_dup} chunk_id duplicados; el índice vectorial requiere clave única."

assert raw_test.filter(F.col("question_id").isNull()).count() == 0, \
    "Hay question_id nulos en test.csv."

print("Lectura validada: sin nulos, sin duplicados, formato coherente.")

## 6. Normalizar `gold_chunk_ids`

En el CSV esta columna viene como texto plano, con esta pinta:

```
[doc_0012_chunk_2, doc_0012_chunk_3]
```

Para poder compararla con las citas del agente necesitas un `ARRAY<STRING>` de verdad. La
estrategia: quitar corchetes y comillas, quitar espacios, y partir por coma.

Nota importante: `gold_chunk_ids` es el **conjunto mínimo** de evidencia. Eso tendrá
consecuencias directas sobre la métrica de precisión más adelante — citar de más te castiga.

In [0]:
def parse_id_list(col):
    """Convierte '[a, b]' (string) en array<string> ['a','b'], tolerante a espacios y comillas."""
    cleaned = F.regexp_replace(col, r"[\[\]'\"]", "")
    cleaned = F.regexp_replace(cleaned, r"\s+", "")
    return F.filter(F.split(cleaned, ","), lambda x: x != "")

train = raw_train.withColumn("gold_chunk_ids", parse_id_list(F.col("gold_chunk_ids")))

display(train.select("question_id", "question", "gold_chunk_ids").limit(5))

### Validar el parsing contra la realidad

Dos comprobaciones. Primero: ningún array quedó vacío. Segundo, y más importante: todos los
`gold_chunk_ids` deben existir realmente en el corpus. Si un ID gold no aparece en
`corpus.csv`, el parsing lo deformó (o el dataset tiene un problema) y ninguna métrica de
citación sería confiable.

In [0]:
empty_gold = train.filter(F.size("gold_chunk_ids") == 0).count()
assert empty_gold == 0, (
    f"{empty_gold} preguntas quedaron sin gold_chunk_ids tras el parsing. "
    "Revisa el formato real de la columna en la celda de inspección cruda (paso 4)."
)

corpus_ids = raw_corpus.select(F.col("chunk_id").alias("gid"))
gold_ids = train.select(F.explode("gold_chunk_ids").alias("gid")).distinct()
orphans = gold_ids.join(corpus_ids, "gid", "left_anti")
n_orphans = orphans.count()

if n_orphans:
    display(orphans.limit(10))
assert n_orphans == 0, (
    f"{n_orphans} gold_chunk_ids no existen en el corpus. El parsing produjo IDs inválidos "
    "o el dataset está incompleto."
)

print(f"Parsing validado: {gold_ids.count():,} chunk_ids gold distintos, todos presentes en el corpus.")

## 7. Escribir las tablas Delta

El corpus necesita **Change Data Feed** habilitado. No es opcional ni decorativo: el índice
Delta Sync de AI Search (notebook 102) usa el CDF para saber qué filas cambiaron y re-embeber
solo esas, en lugar de recalcular todo el índice. Sin CDF, la creación del índice falla.

In [0]:
(raw_corpus.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(T_CORPUS))
(train.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(T_TRAIN))
(raw_test.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(T_TEST))

# CDF: requisito del índice Delta Sync
spark.sql(f"ALTER TABLE {T_CORPUS} SET TBLPROPERTIES (delta.enableChangeDataFeed = true)")

print(f"{T_CORPUS}  (CDF habilitado)")
print(f"{T_TRAIN}")
print(f"{T_TEST}")

## 8. EDA breve

Tres preguntas rápidas sobre los datos, cada una con una consecuencia de diseño.

### 8.1 ¿Qué tipos de documento hay?

`doc_type` es un campo filtrable en el índice vectorial. Saber qué categorías existen te
permite decidir si vale la pena filtrar por tipo durante el retrieval.

In [0]:
display(
    spark.table(T_CORPUS)
    .groupBy("doc_type")
    .agg(F.count("*").alias("n_chunks"), F.countDistinct("doc_id").alias("n_docs"))
    .orderBy(F.desc("n_chunks"))
)

### 8.2 ¿Qué tan largos son los chunks?

El chunking ya viene hecho por la competencia. En un proyecto propio, esta decisión sería
tuya y sería una de las que más impacta la calidad del RAG: chunks muy largos diluyen la
señal del embedding, chunks muy cortos pierden contexto.

Referencia útil: `databricks-gte-large-en` admite una ventana de 8192 tokens, así que
ningún chunk de este dataset se va a truncar.

In [0]:
display(
    spark.table(T_CORPUS)
    .select(F.length("chunk_text").alias("n_chars"))
    .selectExpr(
        "count(*)                       AS n_chunks",
        "round(avg(n_chars))            AS chars_promedio",
        "min(n_chars)                   AS chars_min",
        "percentile_approx(n_chars,0.5) AS chars_mediana",
        "max(n_chars)                   AS chars_max",
    )
)

### 8.3 ¿Cuántos chunks gold tiene cada pregunta?

Esta distribución es la más importante de las tres para lo que viene. Si la mayoría de las
preguntas se responde con **un solo** chunk, entonces un agente que cite 5 chunks va a tener
una precisión de citación pésima aunque acierte la respuesta.

In [0]:
display(
    spark.table(T_TRAIN)
    .groupBy(F.size("gold_chunk_ids").alias("n_gold_chunks"))
    .agg(F.count("*").alias("n_preguntas"))
    .orderBy("n_gold_chunks")
)

## 9. Split dev / holdout

`train.csv` trae las etiquetas gold, y lo vas a usar para dos cosas distintas:

- **dev (80%)** — para iterar: probar valores de `k`, prompts, políticas de citación
- **holdout (20%)** — para medir *una sola vez*, al final, la configuración ganadora

Si mides y ajustas sobre el mismo conjunto, terminas eligiendo la configuración que mejor se
adapta al ruido de esa muestra en particular. Es el mismo overfitting de siempre, solo que
ahora ocurre en la selección de hiperparámetros en vez de en el entrenamiento de un modelo.

La semilla fija (`42`) hace que el split sea idéntico en cada corrida.

In [0]:
dev_df, holdout_df = spark.table(T_TRAIN).randomSplit([0.8, 0.2], seed=SEED)

dev_df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(T_DEV)
holdout_df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(T_HOLDOUT)

print(f"dev     : {spark.table(T_DEV).count():>5,} preguntas  -> {T_DEV}")
print(f"holdout : {spark.table(T_HOLDOUT).count():>5,} preguntas  -> {T_HOLDOUT}")

## 10. Resumen y siguiente paso

Objetos creados en Unity Catalog:

| Tabla | Contenido |
|---|---|
| `agenteval_corpus` | Base de conocimiento en chunks (CDF habilitado) |
| `agenteval_train` | Preguntas con respuesta y evidencia gold |
| `agenteval_test` | Preguntas sin etiquetas (para la submission) |
| `agenteval_train_dev` | 80% de train, para iterar |
| `agenteval_train_holdout` | 20% de train, para la validación final |

Puedes verificarlo en la UI: **Catalog → big_data_ii_2025 → spark_examples → Tables**.

In [0]:
display(spark.sql(f"SHOW TABLES IN {CATALOG}.{SCHEMA} LIKE 'agenteval*'"))